# Day3_02C_Research_Pipeline_Quality_Gate

## OpenAI Agents SDK - Advanced Agent Pattern

**Duration:** 20 minutes  
**Audience:** Engineering Faculty  
**Environment:** VS Code Notebook  
**Theme:** Building a research pipeline with quality checks

---

## Learning Objectives

By the end of this notebook, participants will be able to:

- Understand what a **quality gate** means in Agentic AI.
- Build a simple multi-agent research pipeline.
- Use specialist Agents for research, summarization, and review.
- Add a basic quality check before accepting the final output.
- Relate quality gates to enterprise AI governance and academic review.

---

## Previous Notebook Recap

In the previous notebook, we learned how to run multiple Agents in parallel.

That helped us generate outputs faster.

But speed alone is not enough.

In enterprise and academic systems, we also need quality.

This notebook answers an important question:

> How do we check whether an Agent output is good enough before using it?


# 1. Real-World Story: DSU Research Proposal Review

Imagine a faculty member wants to prepare a short research proposal on:

> Agentic AI for Student Learning Support

A simple AI assistant may generate a proposal quickly.

But before submitting it, we should check:

- Is the topic clear?
- Are the objectives meaningful?
- Are the risks mentioned?
- Is the output classroom/research friendly?
- Is it too vague?
- Does it need improvement?

This checking stage is called a **Quality Gate**.


# 2. What is a Quality Gate?

A **quality gate** is a checkpoint.

It decides whether the output can move forward.

```text
Draft Output
   ↓
Quality Gate
   ↓
Pass → Final Output
Fail → Improve / Rework
```

---

## Simple Examples

| Area | Quality Gate Checks |
|---|---|
| Academic paper | Clarity, structure, references, research gap |
| Software release | Tests passed, security scan passed, code review done |
| AI response | Correctness, completeness, safety, relevance |
| FDP material | Simple language, examples, exercises, teaching flow |

---

## Teaching Analogy

A quality gate is like a faculty review before sending a paper to a journal.


# 3. Architecture for This Notebook

We will build a simple research pipeline.

```text
Research Topic
   ↓
Research Planner Agent
   ↓
Draft Writer Agent
   ↓
Quality Reviewer Agent
   ↓
Quality Gate Decision
   ↓
Final Output or Rework Suggestion
```

This is a very common Agentic AI pattern.


# 4. Setup

We will use:

- `Agent`
- `Runner`
- Python functions for orchestration
- Simple rule-based quality gate

Important:

In VS Code/Jupyter notebooks, use:

```python
await Runner.run(...)
```

not:

```python
Runner.run_sync(...)
```


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

from agents import Agent, Runner

# 5. Load Environment Variables

This uses the standard root `.env` loading pattern used across the FDP notebooks.


In [ ]:
current = Path.cwd().resolve()

for folder in [current] + list(current.parents):
    env = folder / ".env"
    if env.exists():
        load_dotenv(env)
        break

print("API Key Loaded:", os.getenv("OPENAI_API_KEY") is not None)

# 6. Create Specialist Agents

We will create three Agents:

1. **Research Planner Agent**  
   Defines the structure and direction.

2. **Draft Writer Agent**  
   Creates the research proposal draft.

3. **Quality Reviewer Agent**  
   Reviews the draft and gives feedback.


In [ ]:
research_planner_agent = Agent(
    name="Research Planner Agent",
    instructions="""
    You are a research planning assistant for engineering faculty.
    Given a research topic, create a simple research plan with:
    - problem statement
    - possible objectives
    - possible methodology
    - expected outcome

    Keep the output concise and beginner-friendly.
    """,
)

draft_writer_agent = Agent(
    name="Draft Writer Agent",
    instructions="""
    You are a research proposal drafting assistant.
    Write a short academic-style proposal draft based on the given research plan.
    Use clear language suitable for engineering faculty.
    Avoid overclaiming.
    """,
)

quality_reviewer_agent = Agent(
    name="Quality Reviewer Agent",
    instructions="""
    You are a strict but helpful academic reviewer.
    Review research proposal drafts for:
    - clarity
    - completeness
    - relevance
    - feasibility
    - academic tone

    Give specific feedback and mention whether the draft looks ready or needs improvement.
    """,
)

# 7. Step 1 - Create a Research Plan

We start with a topic.

The planner Agent creates the initial research structure.


In [ ]:
topic = "Agentic AI for Student Learning Support in Engineering Colleges"

plan_result = await Runner.run(
    research_planner_agent,
    f"Create a research plan for this topic: {topic}"
)

print(plan_result.final_output)

## What did we just learn?

The planner Agent does not write the full proposal.

It creates the structure.

This is similar to how faculty first prepare an outline before writing a paper.


# 8. Step 2 - Write a Draft Proposal

Now the Draft Writer Agent uses the plan and writes a short proposal.


In [ ]:
draft_result = await Runner.run(
    draft_writer_agent,
    f"""
    Topic: {topic}

    Research Plan:
    {plan_result.final_output}

    Write a short research proposal draft.
    """
)

print(draft_result.final_output)

## What did we just learn?

The second Agent uses the output from the first Agent.

This is a pipeline.

```text
Planner Output → Writer Input
```

Unlike parallel execution, this is sequential because each step depends on the previous step.


# 9. Step 3 - Review the Draft

Now the Quality Reviewer Agent reviews the draft.


In [ ]:
review_result = await Runner.run(
    quality_reviewer_agent,
    f"""
    Please review the following research proposal draft.

    Topic:
    {topic}

    Draft:
    {draft_result.final_output}
    """
)

print(review_result.final_output)

# 10. Basic Quality Gate

For classroom teaching, we will create a simple rule-based quality gate.

It checks whether the draft contains important research proposal sections.

This is not a perfect academic reviewer.

But it teaches the concept clearly.


In [ ]:
def research_quality_gate(draft_text: str) -> dict:
    """
    Simple quality gate for a research proposal draft.

    It checks for key proposal components.
    """

    required_terms = {
        "problem": "Problem statement or problem context",
        "objective": "Research objective",
        "method": "Methodology or method",
        "outcome": "Expected outcome",
        "benefit": "Benefit or contribution"
    }

    draft_lower = draft_text.lower()

    missing_items = []

    for term, description in required_terms.items():
        if term not in draft_lower:
            missing_items.append(description)

    passed = len(missing_items) == 0

    return {
        "passed": passed,
        "missing_items": missing_items,
        "score": len(required_terms) - len(missing_items),
        "max_score": len(required_terms)
    }

# 11. Run the Quality Gate

Now let us check if the draft passes our basic quality gate.


In [ ]:
gate_result = research_quality_gate(draft_result.final_output)

print(gate_result)

## What did we just learn?

The quality gate provides a decision.

It can be:

- Pass
- Fail
- Needs improvement

In real systems, this gate may use:

- LLM reviewers
- rule-based checks
- policy engines
- human approval
- automated scoring


# 12. Combine Reviewer + Quality Gate

Now we will create a complete function.

This function:

1. Creates the research plan.
2. Writes the proposal draft.
3. Reviews the draft.
4. Runs the quality gate.
5. Returns the final decision.


In [ ]:
async def research_pipeline_with_quality_gate(topic: str):
    """
    Complete research pipeline with a simple quality gate.
    """

    plan = await Runner.run(
        research_planner_agent,
        f"Create a research plan for this topic: {topic}"
    )

    draft = await Runner.run(
        draft_writer_agent,
        f"""
        Topic: {topic}

        Research Plan:
        {plan.final_output}

        Write a short research proposal draft.
        """
    )

    review = await Runner.run(
        quality_reviewer_agent,
        f"""
        Review the following research proposal draft.

        Topic:
        {topic}

        Draft:
        {draft.final_output}
        """
    )

    gate = research_quality_gate(draft.final_output)

    if gate["passed"]:
        decision = "PASSED - Proposal draft is ready for faculty review."
    else:
        decision = "NEEDS IMPROVEMENT - Proposal draft should be revised before review."

    return {
        "topic": topic,
        "plan": plan.final_output,
        "draft": draft.final_output,
        "review": review.final_output,
        "quality_gate": gate,
        "decision": decision
    }

# 13. Run the Complete Pipeline

This is the main demo of the notebook.


In [ ]:
pipeline_output = await research_pipeline_with_quality_gate(
    "Agentic AI for Personalized Laboratory Assistance"
)

print("DECISION:", pipeline_output["decision"])
print("\nQUALITY GATE:", pipeline_output["quality_gate"])
print("\n--- REVIEW ---\n")
print(pipeline_output["review"])
print("\n--- DRAFT ---\n")
print(pipeline_output["draft"])

# 14. What Did We Just Build?

We built a simple Agentic AI research pipeline.

```text
Topic
   ↓
Planner Agent
   ↓
Draft Writer Agent
   ↓
Reviewer Agent
   ↓
Quality Gate
   ↓
Decision
```

This is more reliable than asking one Agent to directly produce the final output.


# 15. Why This Matters in Enterprise AI

Quality gates are very important in enterprise systems.

## Example: Banking AI Assistant

Before showing a response to a customer:

- Is the answer compliant?
- Does it expose confidential data?
- Is the tone professional?
- Is the recommendation safe?

## Example: DevOps Agent

Before recommending a deployment:

- Did tests pass?
- Did security scans pass?
- Are rollback steps available?
- Is approval required?

## Example: Academic Assistant

Before generating a research proposal:

- Is the objective clear?
- Is the methodology realistic?
- Is the scope manageable?
- Is there any overclaiming?

Quality gates make AI systems more trustworthy.


# 16. Add a Simple Rework Step

If the quality gate fails, we can ask the Draft Writer Agent to revise the proposal.

This is the beginning of an iterative Agentic workflow.

```text
Draft
   ↓
Quality Gate
   ↓
Fail
   ↓
Revise
   ↓
Review Again
```


In [ ]:
async def revise_if_needed(topic: str, draft: str, gate: dict, review: str):
    """
    Revises the draft if the quality gate fails.
    """

    if gate["passed"]:
        return draft

    revision = await Runner.run(
        draft_writer_agent,
        f"""
        Revise the research proposal draft below.

        Topic:
        {topic}

        Original Draft:
        {draft}

        Reviewer Feedback:
        {review}

        Missing Quality Gate Items:
        {gate["missing_items"]}

        Improve the draft and include the missing components.
        """
    )

    return revision.final_output

# 17. Demonstrate Revision Flow

This cell shows how the pipeline can improve weak output automatically.


In [ ]:
revised_draft = await revise_if_needed(
    pipeline_output["topic"],
    pipeline_output["draft"],
    pipeline_output["quality_gate"],
    pipeline_output["review"]
)

print(revised_draft)

## What did we just learn?

A quality gate can do more than accept or reject.

It can trigger improvement.

This is the foundation for self-improving workflows:

```text
Generate → Review → Improve → Finalize
```


# 18. Classroom Exercise

Create a quality gate for an **FDP Session Plan**.

It should check whether the session plan contains:

- learning objectives
- concept explanation
- demo activity
- exercise
- key takeaways

Then run a session-planning Agent and evaluate the output.


In [ ]:
# Exercise Starter Code

def fdp_session_quality_gate(session_plan: str) -> dict:
    """
    TODO:
    Check whether an FDP session plan contains the required teaching elements.
    """

    required_terms = {
        "learning objective": "Learning objectives",
        "concept": "Concept explanation",
        "demo": "Demo activity",
        "exercise": "Hands-on exercise",
        "takeaway": "Key takeaways"
    }

    # TODO:
    # 1. Convert session_plan to lowercase.
    # 2. Check whether each term is present.
    # 3. Return passed, missing_items, score, max_score.

    return {
        "passed": False,
        "missing_items": [],
        "score": 0,
        "max_score": len(required_terms)
    }

# 19. Challenge Exercise

Extend the research pipeline so that:

1. The quality gate score must be at least 4 out of 5.
2. If the draft fails, the system revises it once.
3. The revised draft is reviewed again.
4. The final output includes both first review and second review.

This is closer to a real academic or enterprise review workflow.


# 20. Common Mistakes

Avoid these mistakes:

1. Assuming first Agent output is final.
2. Not checking for completeness.
3. Creating vague quality gates.
4. Making quality gates too strict for early drafts.
5. Ignoring reviewer feedback.
6. Not explaining why output failed.
7. Forgetting human review for important decisions.

---

## Teaching Tip

For faculty, explain that quality gates are not only technical.

They are also pedagogical, academic, ethical, and governance controls.


# 21. Key Takeaways

In this notebook, we learned:

- A quality gate is a checkpoint before accepting output.
- Research pipelines can be built using multiple specialist Agents.
- Sequential pipelines are useful when each step depends on the previous step.
- Quality gates can check completeness, safety, structure, or compliance.
- Failed outputs can be revised and improved.
- Enterprise AI systems need quality gates for trust and governance.

---

## Final Mental Model

```text
Generate
   ↓
Review
   ↓
Quality Gate
   ↓
Pass → Final
Fail → Revise
```

This pattern is useful for research, teaching, compliance, DevOps, banking, and enterprise AI workflows.
